# 服务器 GPU 运行 —— BC 在 D35 上重跑(正确配置)+ 选路先验闭环天花板

**目的**:
1. M11 BC 在 D35 上重跑,**用 M11 §18 的修好配置 `--no-route-weight --epochs 30`**(逆频加权是把头部压垮的主因;早停用 val overall)。含先验 + no-prior 两版 × 4 编码器。
2. 在修正引擎(D41)上测**修好后** BC 的闭环表现 —— 它能否追平/反超零学习查表(held-out val 上 lookup ≈ +8.7% 延误)。

> ⚠️ 上一轮我误用了 `--epochs 15` 且**漏了 `--no-route-weight`**,跑出的是"修好前"的 BC(no-prior macro 0.53),闭环结论作废。本 notebook 已更正。
> 评估纪律:lookup/BC/RL 都是在 train 上拟合的,**看 held-out `val`** 才干净;`trainval` 对拟合型模型部分 in-sample,仅作参考。

> 本 notebook 含服务器专用路径(hxn886)。`.gitignore` 白名单只放行 `.py`+`CHANGELOG.md`,`.ipynb` 不进 git。GPU: A100-40G。

## 设置(你的服务器前置)

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project

PROJ = '/rds/homes/h/hxn886/ondemand/final_project'
PY = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX = (f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
          f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK. shell cell 用  !{PREFIX} <script> <args>')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1) BC 在 D35 上重跑(M11,修好配置)
`--no-route-weight --epochs 30`。产物:含先验 `bc_<enc>_nw_d35.pt`,no-prior `bc_<enc>_noprior_nw_d35.pt`(`_nw` = no-route-weight)。

In [ ]:
!{PREFIX} scripts/12_train_bc.py mlp deepsets settransformer gnn --epochs 30 --no-route-weight

In [ ]:
!{PREFIX} scripts/12_train_bc.py mlp deepsets settransformer gnn --epochs 30 --no-prior --no-route-weight

## 2) BC 闭环天花板(修正引擎 + 运营日)
把**修好后**的 BC(`_nw`)加进闭环对照,看它相对 held-out lookup(val +8.7%)是否更接近信号员。
**看 `val`(干净 held-out);`trainval` 部分 in-sample、仅参考。**

In [ ]:
!{PREFIX} scripts/24_routing_prior_closed_loop.py --split val \
    --bc-noprior artifacts/reports/bc_deepsets_noprior_nw_d35.pt \
    --bc-prior artifacts/reports/bc_deepsets_nw_d35.pt

In [ ]:
!{PREFIX} scripts/24_routing_prior_closed_loop.py --split trainval \
    --bc-noprior artifacts/reports/bc_deepsets_noprior_nw_d35.pt \
    --bc-prior artifacts/reports/bc_deepsets_nw_d35.pt

### 说明
- BC 报告:`artifacts/reports/bc_<enc>[_noprior]_nw_d35.json`(匹配率),checkpoint 同名 `.pt`。
- 闭环对照:`artifacts/reports/routing_prior_closedloop_<split>_*.json`。
- 交回:各编码器修好后的 macro/overall(尤其 no-prior 是否 ≈0.72 > 查表 0.65),以及 `val` 闭环对照的 `wrote ...json` 文件名。